# Training window comparison (1y / 2y / 3y, long and wide)

The same 252-bar forecast period is predicted with three different training lengths (252, 504 and 756 bars). For each length two XGBoost models are trained: a **long** model (one row per bar and ticker, the portfolio variance comes from the per-ticker forecasts plus an EWMA covariance bank) and a **wide** model (one row per bar, it predicts the portfolio variance directly). VaR is produced for all six runs, and the backtest suite and the graphs are compared side by side.

The forecast window is identical for every run, so the realized losses are identical too: any difference comes from the length of the training data and from the model format.

In [ ]:
from itables import show
import numpy as np
import pandas as pd
import data_management as dm
from core import feature_builders as fb
import tree_models as tm
from graph_helpers import (
    plot_var_vs_loss, plot_var_models_vs_loss, plot_models_output_vs_realized_variance, plot_level_and_volatility,
    var_vs_loss_date_picker, var_models_vs_loss_date_picker, plot_exceedance_days_focus,
)

In [2]:
TICKERS = ['AAPL', 'JPM', 'XOM', 'KO']

WEIGHTS = {
    'AAPL': 0.25,
    'JPM': 0.25,
    'XOM': 0.25,
    'KO': 0.25,
}

df = pd.read_csv(
    './data_general/4_ticker_data.csv',
    parse_dates=['Date'],
    index_col='Date',
)
print(df.index[0].date(), '->', df.index[-1].date(), '|', len(df), 'bars')

2020-01-02 -> 2024-12-31 | 1258 bars


In [3]:
FORECAST_BARS = 252                                   # forecast window: the last 252 bars of the data (same for every model)
TRAIN_BARS = {'1y': 252, '2y': 504, '3y': 756}        # training window lengths in bars, counted back from the forecast start
WARMUP_BARS = 30                                      # bars before train_start used only to warm up rolling windows / EWMA (no first days dropped)

TARGET_SCALE = 10000                                  # the target (r^2) is multiplied by this for training, predictions are divided back
VAR_CONFIDENCE = 0.95                                 # 0.95 -> lower 5% quantile of the standardized residuals
VAR_WINDOW = 252                                      # VaR quantile window: the last N residuals
VAR_MIN_PERIODS = 30                                  # fewest prior residuals a day needs to get a VaR
EWMA_LAM = 0.94                                       # RiskMetrics decay: larger = forgets slower
EWMA_MIN_PERIODS = 30                                 # bars that seed the EWMA covariance

In [4]:
n = len(df)
forecast_start = df.index[n - FORECAST_BARS]
forecast_end = df.index[-1]
train_end = df.index[n - FORECAST_BARS - 1]           # the last bar before the forecast starts

calendars = {}
for name, train_bars in TRAIN_BARS.items():
    train_start = df.index[n - FORECAST_BARS - train_bars]
    calendars[name] = dm.DailyGroundTruthCalendar(
        df, TICKERS, WEIGHTS,
        train_start=train_start, train_end=train_end,
        forecast_start=forecast_start, forecast_end=forecast_end,
        warmup_bars=WARMUP_BARS,
    )

pd.DataFrame({
    name: {
        'train_start': cal.train_data.index[0].date(),
        'train_end': cal.train_data.index[-1].date(),
        'train_bars': len(cal.train_data),
        'forecast_start': cal.forecast_dates[0].date(),
        'forecast_end': cal.forecast_dates[-1].date(),
        'forecast_bars': len(cal.forecast_dates),
    }
    for name, cal in calendars.items()
})

[WARNING] train_start=2022-12-29 falls on Thursday, expected Monday (using it as-is).
[WARNING] forecast_start=2024-01-02 falls on Tuesday, expected Monday (using it as-is).
[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 10 weekday days had no data and were skipped (likely market holidays): 2023-01-02, 2023-01-16, 2023-02-20, 2023-04-07, 2023-05-29 and 5 more
[INFO] train set: 252 bars, 10 weekday days skipped
[WARNING] [forecast] 9 weekday days had no data and were skipped (likely market holidays): 2024-01-15, 2024-02-19, 2024-03-29, 2024-05-27, 2024-06-19 and 4 more
[INFO] forecast set: 252 bars, 9 weekday days skipped
[WARNING] train_start=2021-12-29 falls on Wednesday, expected Monday (using it as-is).
[WARNING] forecast_start=2024-01-02 falls on Tuesday, expected Monday (using it as-is).
[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 19 weekday days had no data and wer

,1y,2y,3y
train_start,2022-12-29,2021-12-29,2020-12-29
train_end,2023-12-29,2023-12-29,2023-12-29
train_bars,252,504,756
forecast_start,2024-01-02,2024-01-02,2024-01-02
forecast_end,2024-12-31,2024-12-31,2024-12-31
forecast_bars,252,252,252


In [5]:
# (TrainingFeature, ForecastFeature) for each calendar
features = {
    name: fb.build_long_features(cal, target_scale=TARGET_SCALE)
    for name, cal in calendars.items()
}

[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] long train: 2016 rows -> 2016 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] long train: 3024 rows -> 3024 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)


In [6]:
# Both banks are SINGLE-USE: training fills them in place, so every training run gets new ones
# (tickers and weights of the EWMA bank are read from the calendar).
models = {}
for name, cal in calendars.items():
    training_feature, _ = features[name]
    models[name] = tm.xgboost_train(
        cal, training_feature,
        residual_bank=dm.ResidualBank(mu_mode='constant', default_mu_hat=0.0),
        ewma_bank=dm.EwmaVarianceBank(cal, ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS),
    )

[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2022-12-29, burn-in ends 2022-12-28, 0 date(s) dropped).
[INFO] train_dates_past_burn_in: 503/503 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2021-12-29, burn-in ends 2021-12-28, 0 date(s) dropped).
[INFO] train_dates_past_burn_in: 755/755 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2020-12-29, burn-in ends 2020-12-28, 0 date(s) dropped).


In [7]:
# Fills copies of each model's banks with the forecast window (no VaR yet), then reads the VaR off the filled bank.
residual_banks, var_dfs = {}, {}
for name, cal in calendars.items():
    _, forecast_feature = features[name]
    residual_banks[name], _ = models[name].calculate_multi_day_standart_residuals_from_long_format_model(
        calendar=cal,
        long_forecast_feature=forecast_feature,
    )
    var_dfs[name] = cal.var_from_residual_bank_forecast_period(
        residual_banks[name],
        var_confidence=VAR_CONFIDENCE,
        multi_day_var_calculation_window_size=VAR_WINDOW,
        run_key=f'xgboost_long_{name}',
        multi_day_var_min_periods=VAR_MIN_PERIODS,
    )

# distribution of the VaR level (varForecast) per model
pd.DataFrame({name: v['varForecast'].astype(float) for name, v in var_dfs.items()}).describe()

/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2024-05-09 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2024-12-27 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-standalone/core/EwmaVarianceBank.p

,1y,2y,3y
count,252.000000,252.000000,252.000000
mean,0.011678,0.010901,0.011016
std,0.002710,0.002167,0.001924
min,0.007376,0.007059,0.007121
25%,0.009901,0.009405,0.009680
50%,0.010960,0.010594,0.010838
75%,0.012705,0.012062,0.012020
max,0.026792,0.020981,0.020354


In [8]:
# Wide format: one model predicts the PORTFOLIO variance directly (no per-ticker model, no EWMA bank).
# Same calendars, so same training windows and the same forecast period as the long models above.
features_wide = {
    name: fb.build_wide_features(cal, target_scale=TARGET_SCALE)
    for name, cal in calendars.items()
}

models_wide = {}
for name, cal in calendars.items():
    training_feature, _ = features_wide[name]
    models_wide[name] = tm.xgboost_train(
        cal, training_feature,
        residual_bank=dm.ResidualBank(mu_mode='constant', default_mu_hat=0.0),
    )

[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)


In [9]:
# Forecast window into a copy of each wide model's residual bank (no VaR yet), then the VaR is read off the filled bank.
residual_banks_wide, var_dfs_wide = {}, {}
for name, cal in calendars.items():
    _, forecast_feature = features_wide[name]
    residual_banks_wide[name] = models_wide[name].calculate_multi_day_standart_residuals_from_wide_format_model(
        calendar=cal,
        forecast_feature=forecast_feature,
    )
    var_dfs_wide[name] = cal.var_from_residual_bank_forecast_period(
        residual_banks_wide[name],
        var_confidence=VAR_CONFIDENCE,
        multi_day_var_calculation_window_size=VAR_WINDOW,
        run_key=f'xgboost_wide_{name}',
        multi_day_var_min_periods=VAR_MIN_PERIODS,
    )

pd.DataFrame({name: v['varForecast'].astype(float) for name, v in var_dfs_wide.items()}).describe()

,1y,2y,3y
count,252.000000,252.000000,252.000000
mean,0.011790,0.012525,0.011877
std,0.002869,0.003097,0.002410
min,0.003368,0.007879,0.006735
25%,0.009741,0.010306,0.010317
50%,0.011662,0.011674,0.011701
75%,0.013838,0.014199,0.013109
max,0.019333,0.034910,0.026800


In [10]:
# Every run in one place, keyed '<training length>_<model type>' (1y_long, 1y_wide, 2y_long, ...)
runs = {}
for name, cal in calendars.items():
    runs[f'{name}_long'] = {'calendar': cal, 'var_df': var_dfs[name], 'residual_bank': residual_banks[name]}
    runs[f'{name}_wide'] = {'calendar': cal, 'var_df': var_dfs_wide[name], 'residual_bank': residual_banks_wide[name]}
list(runs)

['1y_long', '1y_wide', '2y_long', '2y_wide', '3y_long', '3y_wide']

In [11]:
# Backtest suite, each run as one window (252 days). var_confidence must be the one the VaR was produced with.
backtest = pd.DataFrame({
    key: dm.backtest_forecast_period(run['calendar'], run['var_df'], VAR_CONFIDENCE)
    for key, run in runs.items()
})
backtest

,1y_long,1y_wide,2y_long,2y_wide,3y_long,3y_wide
var_confidence,0.95,0.95,0.95,0.95,0.95,0.95
test_confidence,0.95,0.95,0.95,0.95,0.95,0.95
n_days,252,252,252,252,252,252
n_exceedances,14,12,14,13,14,12
exceedance_rate,0.055556,0.047619,0.055556,0.051587,0.055556,0.047619
expected_exceedance_rate,0.05,0.05,0.05,0.05,0.05,0.05
binomial_pass,True,True,True,True,True,True
traffic_light_zone,1,1,1,1,1,1
kupiec_pof_stat,0.158298,0.030539,0.158298,0.013235,0.158298,0.030539
kupiec_pof_critical_value,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459


In [12]:
# Summary: exceedances, whether the tests passed, and the mean VaR level
summary_rows = [
    'n_exceedances', 'exceedance_rate', 'expected_exceedance_rate', 'traffic_light_zone',
    'kupiec_pof_stat', 'kupiec_pof_critical_value', 'kupiec_pof_pass',
    'christoffersen_ind_pass', 'christoffersen_cc_pass',
]
summary = backtest.loc[summary_rows].copy()
summary.loc['mean_varForecast'] = [runs[key]['var_df']['varForecast'].astype(float).mean() for key in summary.columns]
summary

,1y_long,1y_wide,2y_long,2y_wide,3y_long,3y_wide
n_exceedances,14,12,14,13,14,12
exceedance_rate,0.055556,0.047619,0.055556,0.051587,0.055556,0.047619
expected_exceedance_rate,0.05,0.05,0.05,0.05,0.05,0.05
traffic_light_zone,1,1,1,1,1,1
kupiec_pof_stat,0.158298,0.030539,0.158298,0.013235,0.158298,0.030539
kupiec_pof_critical_value,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459
kupiec_pof_pass,True,True,True,True,True,True
christoffersen_ind_pass,True,True,True,True,True,True
christoffersen_cc_pass,True,True,True,True,True,True
mean_varForecast,0.011678,0.01179,0.010901,0.012525,0.011016,0.011877


In [13]:
# Level and volatility suite (descriptive, no hypothesis tests): how HIGH is the VaR and how much does it MOVE.
# mean_tick_loss is the headline score (lower is better) = exceedance part (VaR too low) + overshoot part (VaR too high).
level_volatility = pd.DataFrame({
    key: dm.level_and_volatility_forecast_period(run['calendar'], run['var_df'], VAR_CONFIDENCE)
    for key, run in runs.items()
})
level_volatility

,1y_long,1y_wide,2y_long,2y_wide,3y_long,3y_wide
var_confidence,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000
n_days,252.000000,252.000000,252.000000,252.000000,252.000000,252.000000
n_exceedances,14.000000,12.000000,14.000000,13.000000,14.000000,12.000000
exceedance_rate,0.055556,0.047619,0.055556,0.051587,0.055556,0.047619
expected_exceedance_rate,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
mean_tick_loss,0.000925,0.000885,0.000903,0.000893,0.000892,0.000882
tick_loss_exceedance_part,0.000282,0.000238,0.000298,0.000210,0.000281,0.000231
tick_loss_overshoot_part,0.000644,0.000647,0.000606,0.000682,0.000611,0.000651
mean_var,0.011678,0.011790,0.010901,0.012525,0.011016,0.011877
median_var,0.010960,0.011662,0.010594,0.011674,0.010838,0.011701


In [14]:
# The same table as small multiples: every measure on its own axis, runs side by side (blue = long, orange = wide)
#plot_level_and_volatility(level_volatility, title='VaR level and volatility (252-day forecast, 1y / 2y / 3y training)')

In [15]:
# Do the models miss on the same days? One row per day where at least one run has loss > VaR.
# n_models = how many runs exceeded that day (6 = all of them, 1 = only one run).
exceedance_days = dm.exceedance_days_by_model({key: run['var_df'] for key, run in runs.items()})
show(exceedance_days)

In [16]:
# How much do the runs agree? Number of days by how many runs missed them.
model_cols = [c for c in exceedance_days.columns if c not in ('loss', 'n_models')]
exceedance_days['n_models'].value_counts().sort_index().rename_axis('runs that missed the day').to_frame('days')

,days
runs that missed the day,
1,1
2,3
3,1
4,3
5,3
6,7


In [17]:
# Per (day, model): realized loss, VaR forecast and their difference (loss - VaR, positive = the VaR was exceeded by that much).
exceedance_details = dm.exceedance_day_details({key: run['var_df'] for key, run in runs.items()}, exceedance_days)
show(exceedance_details.reset_index())

In [18]:
# All exceedance days stacked in ONE figure: per day a graph (every run's VaR line + the realized loss) and,
# next to it, a table with loss, VaR and loss - VaR per run (exceeded rows in red). Pass fewer days for a shorter figure.
plot_exceedance_days_focus(
    {key: run['var_df'] for key, run in runs.items()},
    exceedance_days,
    window_bars=10,
)

In [19]:
# The VaR of every run (long and wide, 1y / 2y / 3y) on top of the same realized loss
plot_var_models_vs_loss(
    {key: (run['var_df'], run['residual_bank']) for key, run in runs.items()},
    title='VaR Forecast vs Realized Loss (long / wide, 1y / 2y / 3y training)',
    calendar=calendars['3y'],
)

In [20]:
# Same graph with a date picker (needs ipywidgets): pick a start date and a window length, the graph jumps there.
var_models_vs_loss_date_picker(
    {key: (run['var_df'], run['residual_bank']) for key, run in runs.items()},
    title='VaR Forecast vs Realized Loss (long / wide, 1y / 2y / 3y training)',
    calendar=calendars['3y'],
)

In [21]:
# Model outputs (predicted_variance) against the EWMA realized variance, forecast window only
plot_models_output_vs_realized_variance(
    calendars['3y'],
    {key: run['residual_bank'] for key, run in runs.items()},
    only_forecast_window=True,
)

In [22]:
# One graph per run (hover shows predicted_variance, predicted_sigma, mu_hat and the standardized residual)
for key, run in runs.items():
    plot_var_vs_loss(run['var_df'], run['residual_bank'], title=f'VaR vs Realized Loss ({key})', calendar=run['calendar']).show()